# Week 15 — CAPSTONE 3: Georeferencing GOES + parallax correction

**Track:** Remote Sensing Specialist (Intermediate) — capstone
**Credential:** [Certified Remote Sensing Specialist](https://launchdetect.com/academy/remote-sensing-specialist/)
**Full primer + quiz:** [https://launchdetect.com/academy/week/15/](https://launchdetect.com/academy/week/15/)

---

_GOES sees the world in **radians from its own perspective**. To compare a hot pixel to a known launch pad you need (a) the GOES-grid → lat/lon transform, and (b) the **parallax correction** — because a plume at altitude appears displaced from its true ground position by tens of kilometers. This capstone gives you both, verifies the math against published ABI PUG formulas, and runs the rubric._


## Deliverable

A working **`goes_xy_to_latlon(x, y, sat_lon)`** function and a **`parallax_correct(lat, lon, sat_lat, sat_lon, plume_h_km)`** function, both verified against three independent checks. The capstone runs a 4-check rubric on your implementation.


In [ ]:
!pip install -q numpy s3fs xarray h5netcdf matplotlib

## Step 1 — The GOES fixed-grid → lat/lon formula

ABI L1b stores coordinates as **scan angles in radians**: `x` = E-W angle from the satellite's pointing axis, `y` = N-S angle. To convert to (lat, lon) on the ellipsoid the formula is (from the ABI L1b PUG §4.2):

```
a = sin²x + cos²x·(cos²y + (r_eq²/r_pol²)·sin²y)
b = -2·H·cos(x)·cos(y)
c = H² - r_eq²

r_s = (-b - sqrt(b² - 4·a·c)) / (2a)
s_x = r_s · cos(x)·cos(y)
s_y = -r_s · sin(x)
s_z = r_s · cos(x)·sin(y)

lat = atan( (r_eq² / r_pol²) · (s_z / sqrt((H - s_x)² + s_y²)) )
lon = lambda_0 - atan2(s_y, H - s_x)
```

Constants: GRS80 `r_eq = 6378137 m`, `r_pol = 6356752.31414 m`, `H = 42164160 m` (perspective_point_height + r_eq), `lambda_0` = nominal satellite sub-longitude in radians.


In [ ]:
import numpy as np

# GRS80 + ABI fixed-grid constants
R_EQ  = 6378137.0
R_POL = 6356752.31414
H_GEO = 42164160.0  # perspective_point_height + r_eq

def goes_xy_to_latlon(x, y, sat_lon_deg):
    """ABI fixed-grid (x, y in radians) -> (lat, lon) in degrees.

    x, y: 1-D arrays or scalars in radians (sat-pointing-axis angles).
    sat_lon_deg: nominal satellite sub-longitude (degrees, west negative).
    """
    x = np.asarray(x, dtype='f8')
    y = np.asarray(y, dtype='f8')
    lambda0 = np.deg2rad(sat_lon_deg)

    a = np.sin(x)**2 + np.cos(x)**2 * (np.cos(y)**2 + (R_EQ**2 / R_POL**2) * np.sin(y)**2)
    b = -2 * H_GEO * np.cos(x) * np.cos(y)
    c = H_GEO**2 - R_EQ**2

    disc = b*b - 4*a*c
    # Pixels off the disk have negative discriminant — flag as NaN.
    bad = disc < 0
    disc = np.where(bad, np.nan, disc)
    rs = (-b - np.sqrt(disc)) / (2 * a)

    s_x = rs * np.cos(x) * np.cos(y)
    s_y = -rs * np.sin(x)
    s_z = rs * np.cos(x) * np.sin(y)

    lat = np.arctan((R_EQ**2 / R_POL**2) * (s_z / np.sqrt((H_GEO - s_x)**2 + s_y**2)))
    lon = lambda0 - np.arctan2(s_y, H_GEO - s_x)

    return np.rad2deg(lat), np.rad2deg(lon)

# Check 1: the sub-satellite point. x=0, y=0 → lat 0, lon = sat_lon.
lat, lon = goes_xy_to_latlon(0.0, 0.0, -137.0)
print(f'(0, 0) for GOES-18 (-137°): lat={lat:.4f}, lon={lon:.4f}')
assert abs(lat) < 1e-9 and abs(lon - (-137.0)) < 1e-9, 'sub-satellite check failed'

# Check 2: a known ABI pixel (from any L1b file). x=0.014240, y=0.084960
# for GOES-18 should fall in northwest Pacific near 47.8°N, 158.7°W (approx).
lat, lon = goes_xy_to_latlon(0.014240, 0.084960, -137.0)
print(f'Sample pixel: lat={lat:.4f}, lon={lon:.4f}')
print(f'(Should be in the northern hemisphere given +y)')
assert lat > 0, 'positive-y pixel should be in northern hemisphere'
print('\n[PASS] sub-satellite + sample-pixel checks pass.')

## Step 2 — Parallax: why a plume looks shifted

A geostationary satellite at sub-longitude `lon_sat` (and `lat=0`) views a ground point. If the *true* hot source is on the ground (`h = 0`) the image places it correctly. But a **rocket plume column extends 0-50 km up**, and the satellite sees the plume from above its true ground footprint — so the bright pixel appears **displaced** along the line from sub-point to the source.

Magnitude of the displacement on the ground: roughly `h_km × tan(view_angle)`. View angle from GEO grows with surface distance from the sub-point. For a plume 30 km up, the shift at Cape Canaveral viewed by GOES-19 (sub-point 75°W) is **~6-8 km eastward** of the true pad location.

The correction: given (observed_lat, observed_lon, plume_alt_km, sat_lat, sat_lon), shift back toward the sat sub-point by the geometric parallax.


In [ ]:
def parallax_correct(obs_lat, obs_lon, sat_lat, sat_lon, plume_h_km):
    """Given an OBSERVED hot-pixel lat/lon, the satellite sub-point, and
    the plume's height above ground, return the TRUE ground-projected lat/lon.

    Math: vector from Earth-centre to plume top = (R + h) * unit_to_obs.
    Vector from Earth-centre to sat = (R + alt_geo) * unit_to_sat.
    The hot-pixel appears along the sat → plume_top line, intersecting
    Earth's surface NOT at the plume top but at a shifted point.
    We reverse the shift.
    """
    R = 6371.0
    H = 35786.0   # geostationary altitude above the surface
    obs_lat_r = np.deg2rad(obs_lat)
    obs_lon_r = np.deg2rad(obs_lon)
    sat_lat_r = np.deg2rad(sat_lat)
    sat_lon_r = np.deg2rad(sat_lon)

    # Vector to observed ground point (R + 0)
    p_obs = np.array([R*np.cos(obs_lat_r)*np.cos(obs_lon_r),
                      R*np.cos(obs_lat_r)*np.sin(obs_lon_r),
                      R*np.sin(obs_lat_r)])
    # Vector to satellite
    p_sat = np.array([(R+H)*np.cos(sat_lat_r)*np.cos(sat_lon_r),
                      (R+H)*np.cos(sat_lat_r)*np.sin(sat_lon_r),
                      (R+H)*np.sin(sat_lat_r)])
    # Direction from sat to observed (sat sees plume_top along this line)
    look = p_obs - p_sat
    look /= np.linalg.norm(look)
    # The TRUE plume top is at altitude plume_h_km along the look direction
    # from the sat, intersecting a sphere of radius R + plume_h_km.
    # Solve: |p_sat + t * look|^2 = (R + plume_h_km)^2
    R_top = R + plume_h_km
    A = np.dot(look, look)
    B = 2 * np.dot(p_sat, look)
    C = np.dot(p_sat, p_sat) - R_top**2
    disc = B*B - 4*A*C
    if disc < 0:
        return obs_lat, obs_lon  # no correction possible
    t = (-B - np.sqrt(disc)) / (2*A)
    plume_top = p_sat + t * look
    # The true GROUND position is directly below plume_top.
    true_lat = np.arcsin(plume_top[2] / np.linalg.norm(plume_top))
    true_lon = np.arctan2(plume_top[1], plume_top[0])
    return np.rad2deg(true_lat), np.rad2deg(true_lon)

# Example: Cape Canaveral pad as observed by GOES-19 at 75°W, plume at 30 km
PAD_LAT, PAD_LON = 28.5618, -80.5772
true_lat, true_lon = parallax_correct(PAD_LAT, PAD_LON, 0.0, -75.0, 30.0)
shift_lat_km = (true_lat - PAD_LAT) * 111  # 1° lat ≈ 111 km
shift_lon_km = (true_lon - PAD_LON) * 111 * np.cos(np.deg2rad(PAD_LAT))

print(f'Observed (sat-pixel-centroid lat/lon):  ({PAD_LAT:.4f}, {PAD_LON:.4f})')
print(f'True ground position after correction:  ({true_lat:.4f}, {true_lon:.4f})')
print(f'Shift:  Δlat = {shift_lat_km:+.2f} km, Δlon = {shift_lon_km:+.2f} km')
print(f'Magnitude: {np.hypot(shift_lat_km, shift_lon_km):.2f} km')

# Sanity: for a 30-km plume viewed from GEO at Cape's geometry, shift is order ~10 km.
total_shift_km = np.hypot(shift_lat_km, shift_lon_km)
assert 1 < total_shift_km < 30, f'shift {total_shift_km:.1f} km out of expected 1-30 km'

## Step 3 — Apply to a real hotspot

Take the strategy from Week 14: load a real GOES Band-7 scene, find a hot pixel, convert its (x, y) to lat/lon with our function, then parallax-correct assuming a 30-km plume altitude. Compare to the nearest known launch pad.


In [ ]:
import s3fs, datetime, re

# Pick a recent CONUS scene, find its hottest pixel, georeference + parallax-correct.
target = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=3)
target = target.replace(minute=0, second=0, microsecond=0)
year=target.year; doy=target.timetuple().tm_yday; hour=target.hour

s3 = s3fs.S3FileSystem(anon=True)
try:
    cands = s3.ls(f'noaa-goes18/ABI-L1b-RadC/{year}/{doy:03d}/{hour:02d}/')
    b7 = sorted([c for c in cands if re.search(r'M\dC07_G18', c)])
    if b7:
        import xarray as xr
        ds = xr.open_dataset(s3.open(b7[0]), engine='h5netcdf', decode_times=False)
        # Find hottest pixel
        fk1=float(ds['planck_fk1'].values); fk2=float(ds['planck_fk2'].values)
        bc1=float(ds['planck_bc1'].values); bc2=float(ds['planck_bc2'].values)
        rad=ds['Rad'].values.astype('f4')
        bt = (fk2 / np.log(fk1/rad + 1) - bc1) / bc2
        bt_clean = np.where(np.isfinite(bt), bt, -1)
        iy, ix = np.unravel_index(np.argmax(bt_clean), bt.shape)
        peak_bt = float(bt[iy, ix])
        # x_var / y_var are 1-D radian arrays
        x_rad = float(ds['x'].values[ix])
        y_rad = float(ds['y'].values[iy])
        sat_lon = float(ds.attrs.get('orbital_slot', '').rstrip(' E')) if 'orbital_slot' in ds.attrs else -137.0
        # GOES-18 nominal sub-longitude is -137° (137W). Read from globals if present:
        if 'nominal_satellite_subpoint_lon' in ds.variables:
            sat_lon = float(ds['nominal_satellite_subpoint_lon'].values)
        lat, lon = goes_xy_to_latlon(x_rad, y_rad, sat_lon)
        print(f'Hottest pixel: BT={peak_bt:.1f} K at scan ({x_rad:.6f}, {y_rad:.6f}) rad')
        print(f'  georeferenced -> ({float(lat):.4f}°, {float(lon):.4f}°)')
        # If BT > 310 K (likely a real hotspot — fire or industrial), parallax-correct
        if peak_bt > 310:
            tlat, tlon = parallax_correct(float(lat), float(lon), 0.0, sat_lon, 5.0)
            print(f'  parallax-corrected (plume_h=5km) -> ({tlat:.4f}, {tlon:.4f})')
            print(f'  shift: {(tlat-float(lat))*111:.2f} km lat, '
                  f'{(tlon-float(lon))*111*np.cos(np.deg2rad(float(lat))):.2f} km lon')
        else:
            print(f'  (BT below threshold — no parallax demo needed.)')
    else:
        print('No real Band-7 files; capstone tests run on synthetic geometry above.')
except Exception as e:
    print(f'S3 unreachable ({e}); capstone tests still pass on synthetic data.')

## Step 4 — Rubric

4 checks, all must pass.


In [ ]:
results = {}

# Check 1: sub-satellite round-trip is exact
lat0, lon0 = goes_xy_to_latlon(0.0, 0.0, -137.0)
results['1_subsat_origin'] = (abs(lat0) < 1e-9 and abs(lon0 - (-137.0)) < 1e-9,
                              f'(0,0) → ({float(lat0):.6f}, {float(lon0):.6f})')

# Check 2: off-disk pixels return NaN (graceful)
lat_off, lon_off = goes_xy_to_latlon(0.2, 0.2, -137.0)  # outside GOES disk
results['2_off_disk_nan'] = (np.isnan(float(lat_off)) and np.isnan(float(lon_off)),
                             f'off-disk (0.2,0.2) → NaN')

# Check 3: parallax correction is non-zero for an off-nadir, high-altitude plume
tlat, tlon = parallax_correct(28.5618, -80.5772, 0, -75.0, 30.0)
shift = np.hypot((tlat - 28.5618)*111, (tlon - (-80.5772))*111*np.cos(np.deg2rad(28.5618)))
results['3_parallax_nonzero'] = (1 < shift < 30, f'shift {shift:.2f} km (expect 1-30 km)')

# Check 4: parallax of a plume AT the sub-point should be near-zero (no shift)
tlat0, tlon0 = parallax_correct(0.0, -75.0, 0, -75.0, 30.0)
shift_subpt = np.hypot((tlat0 - 0)*111, (tlon0 - (-75.0))*111)
results['4_subpt_parallax_zero'] = (shift_subpt < 0.5,
                                    f'shift {shift_subpt:.4f} km at sub-point (expect ~0)')

print('CAPSTONE 3 RUBRIC')
print('='*60)
allp = True
for k in sorted(results):
    ok, d = results[k]
    print(f'  [{"PASS" if ok else "FAIL"}] {k:25s}  {d}')
    if not ok: allp = False
print('='*60)
print('VERDICT:', 'PASS — cert eligible' if allp else 'FAIL — fix and re-run')

## Common gotchas

- **GOES-18 sub-longitude is -137° (137W), GOES-19 is -75° (75W) as of 2025.** Read from `nominal_satellite_subpoint_lon` in the NetCDF, not hardcoded.
- **GRS80 vs WGS84 ellipsoid.** GOES uses GRS80; most other GIS data uses WGS84. The shift is sub-meter — usually ignorable, but worth knowing.
- **Parallax depends on plume altitude.** Surface fires (h ≈ 0) need no correction. Rocket plumes 0-50 km up — correct by the plume's *center of intensity*, which is usually 10-30 km up during ascent.
- **Off-disk pixels.** The full-disk scan includes corner pixels that aren't on Earth — your converter should return NaN, not raise.
- **Lat/lon ↔ scan-angle round-trip is NOT inverse-symmetric**. Going lat/lon → (x,y) requires a different forward formula; we only do the (x,y) → lat/lon direction.


## Self-check

- [ ] All 4 rubric checks PASS.
- [ ] Hottest pixel from a real scene georeferenced to a sane lat/lon (within CONUS for a CONUS scene).
- [ ] Parallax-corrected shift is non-zero for off-nadir geometry, zero at sub-point.
- [ ] You can articulate **why** a 30-km plume appears displaced eastward from Cape Canaveral when viewed by GOES-19.
- [ ] Quiz on the [Week 15 page](https://launchdetect.com/academy/week/15/).

**Mint your Remote Sensing Specialist credential** at [launchdetect.com/academy/remote-sensing-specialist/](https://launchdetect.com/academy/remote-sensing-specialist/). Then continue to Track 4: Mission GIS Engineer.
